# A humanoid at the ping-pong table

This notebook puts a Unitree G1 humanoid, holding HIM's red paddle, at the HIM table in Isaac Lab, with the calibrated
ball physics. You run it cell by cell on a cloud GPU and see each result as you go. It has two parts:

1. **The template scene**: you drive the robot with your own actions.
2. **The trained policy**: the G1 rallies against a model of HIM's opponent, then plays with the ball hidden.

New to this project? Read the [README](README.md) first. Its glossary defines every term used here.

**Before you start.** A notebook needs a *session*: a cloud machine running this project's `robot` service (Isaac Lab
on one GPU). In a terminal on your computer, from `ping-pong/`, start one and open JupyterLab in it, then open this
notebook there:

```bash
uv run antioch session new --profile robot
uv run antioch jupyter lab
```

Keep `jupyter lab` running while you work: it keeps copying your local edits to the session. After editing a file in
`lib/`, restart the kernel to load the new version.

**About the ball.** The ball isn't a PhysX body. It's the calibrated ball physics (`docs/physics.md`), flown in 1 ms
sub-steps against the robot's real paddle every 5 ms physics step, and pushing back on the paddle when they meet.

## 1. Start Isaac Lab

Start Isaac Lab once per kernel (the Python process that runs this notebook's cells). Each section below builds its
scene, then closes it before the next.

In [ ]:
import os

import antioch
import torch

from lib.config import ROBOT_SIMULATION_CONFIG
from lib.robots import UNITREE_G1
from lib.robots.table_scene import build_table_scene

if os.environ.get("ANTIOCH_ENGINE") != "isaac-lab-3.0":
    raise RuntimeError("open this notebook from `antioch session new --profile robot`")
antioch.start_simulation(ROBOT_SIMULATION_CONFIG)

## 2. The template scene

`build_table_scene` builds the robot from its spec (`lib/robots/unitree_g1.py`) in parallel worlds, called *envs*.
Each env has:

- **the robot:** its 3D model with the hand removed and the paddle fixed to its wrist, every link colliding as its
  URDF says, and its motors and PD gains (how stiffly and with how much damping each motor holds its target);
- **the table and net**, as the robot meets them;
- **the calibrated ball.**

Here the pelvis is pinned (`fixed_base=True`): with no balance controller, a G1 holding its pose would fall.
Set `fixed_base=False` once your controller balances; a fallen robot is then reset to its stance.

Building a scene compiles the GPU ball physics first, which takes about four minutes.

In [ ]:
scene = build_table_scene(UNITREE_G1, num_envs=4, fixed_base=True, reset_spread=False)
observation = scene.reset_all()
print(f"{scene.num_envs} envs, {len(scene.joint_names)} joints, observation {tuple(observation.shape)}, policy step {scene.step_dt} s")
scene.joint_names[:6]

## 3. Step your own actions

An *action* is one number per joint, in the order of `scene.joint_names`. Each joint's target angle is
`default_pose + action_scale * action`, and its PD gains drive it there. Zeros hold the ready stance. The scale is a
quarter of the joint's peak torque over its stiffness: 0.156 rad for the arm joints, so an action of 3.0 moves an arm
joint about 0.5 rad.

Below, the right arm swings while one of HIM's recorded feeds comes in. Replace `my_controller` with yours: it gets
the latest *observation* (one row of numbers per env, describing the robot and the ball) and the time in seconds.
`observation_terms` in `lib/robots/table_scene.py` lists the observation's columns in order; for the G1, columns
109-111 are the ball's position relative to the pelvis.

In [ ]:
from lib.robots.template import RECORDED_FEEDS

swing = scene.joint_names.index("right_shoulder_pitch_joint")


def my_controller(observation: torch.Tensor, t: float) -> torch.Tensor:
    actions = scene.hold_default_pose()
    actions[:, swing] = 3.0 * torch.sin(torch.tensor(2.0 * torch.pi * t))  # about 0.5 rad either way
    return actions


coupling = scene.coupling
rows = torch.arange(scene.num_envs, device=coupling.device)
coupling.launch(rows, torch.tensor(RECORDED_FEEDS[0], dtype=coupling.dtype, device=coupling.device).expand(scene.num_envs, 9))
for step in range(100):  # 2 s
    observation = scene.step(my_controller(observation, step * scene.step_dt))
ball = coupling.ball[0].cpu()
print(f"after 2 s: ball in play {bool(coupling.active[0])}, at {ball[:3].numpy().round(3)} m (table frame)")
print(f"robot-blade contacts so far: {coupling.counters.contacts}")

In [ ]:
scene.close()

## 4. The trained policy

The trained G1 policy rallies against HIM's opponent, a model learnt from the recordings. `docs/policy.md` covers it
in depth.

- **The actor:** `load_rally_actor` fetches the trained network (the *checkpoint*) from the asset store, and folds it
  to the 146 inputs it reads at play.
- **The rallies:** `CoupledRally` plugs into the scene. It hands the ball physics the opponent's paddle every physics
  step, opens rallies with the opponent's recorded feeds, and adds the ball forecast to the observation.

Ten seconds at 64 envs. In the result, a *receive* is a ball the opponent sends toward the robot's half, and a legal
return is a hit whose first bounce lands on the opponent's half. `legal_per_receive` is the headline, with its 95 %
confidence interval; `hit_rate`, rally length in legal shots, and falls per 10 s fill in the picture.

In [ ]:
from lib.assets import G1_RALLY_POLICY
from lib.policy.actor import hide_ball, load_rally_actor
from lib.policy.grade import rally_metrics
from lib.policy.rally import CoupledRally
from lib.policy.run import PUSH_SPEED_MPS

rally = CoupledRally(seed=0, forecast_through_serve=True)
scene = build_table_scene(UNITREE_G1, 64, hooks=rally, opponent=True, push_speed_mps=PUSH_SPEED_MPS)
actor = load_rally_actor(G1_RALLY_POLICY, default_pose=UNITREE_G1.default_pose, device=scene.env.device)
played = rally_metrics(rally.play(scene, actor, 10.0))
{key: played[key] for key in ("receives", "legal_per_receive", "legal_per_receive_ci95", "hit_rate", "rally_length_legal_shots", "falls_per_10s")}

## 5. The blind control

The same network, with the ball and its forecast hidden from it. A policy that really plays from the ball must
collapse. This is a *negative control*: a run that must fail, which proves the measurement above can tell a robot
that sees the ball from one that doesn't.

In [ ]:
blind = rally_metrics(rally.play(scene, hide_ball(actor), 10.0))
{key: blind[key] for key in ("receives", "legal_per_receive", "hit_rate")}

In [ ]:
scene.close()

## 6. The recorded scenarios

The same scenes also run as recorded scenario runs, with their checks, telemetry and the 24 fps film of env 0 in the
HIM lab twin. Run them from a terminal on your computer, in `ping-pong/`:

```bash
uv run antioch scenario run load_humanoid_at_table --cases unitree-g1
uv run antioch scenario run play_ping_pong_policy --cases g1-rally-smoke
```

When you're done, run the cell below to close Isaac. Then, from your terminal, stop JupyterLab and release the
session, so it stops using your organization's compute:

```bash
uv run antioch jupyter stop
uv run antioch session release
```

In [ ]:
antioch.application().close()